# Estabilidade longitudinal — análise agrupada

Reanalisa `ablation_results_all_mrmr.csv` **sem retreinar**.

| Elemento | Significado |
|----------|-------------|
| **Eixo X** | T1, T2, T3 |
| **Eixo Y** | % incidência nas avaliações externas |
| **Cada linha** | um biomarcador (`L \| gm_norm`, etc.) |

Funções: `feature_freq_table_grouped`, `plot_temporal_stability_lines` em `ablation_analysis.py`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from ablation_analysis import (
    feature_freq_table,
    feature_freq_table_grouped,
    filter_ablation_config,
    n_repeats_in_df,
    plot_compare_stability,
    plot_temporal_stability_lines,
    prepare_ablation_df,
)

MODALITY = "vol"  # vol | shape | texture | disp | all
RESULTS_DIR = Path(f"csvs/longitudinal_4_groups/ablation_results/{MODALITY}")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all_mrmr.csv"

TASK = "smci_pmci"
MODEL = "svm"  # svm | rf | mlp
WITH_COMBAT = True
SELECTION_MODE = "mrmr"
EXPECTED_FOLDS = 5  # outer folds do nested CV

MIN_COVERAGE_PCT = 70  # tabela: atributos principais
SAVE_OUTPUTS = True


## 1. Carregar resultados


In [ ]:
df_ablation = prepare_ablation_df(pd.read_csv(RESULTS_PATH))
n_repeats = n_repeats_in_df(df_ablation)

df_cfg = filter_ablation_config(
    df_ablation,
    task=TASK,
    modality=MODALITY,
    model_key=MODEL,
    with_combat=WITH_COMBAT,
    selection_mode=SELECTION_MODE,
    expected_folds=EXPECTED_FOLDS,
)
n_runs = len(df_cfg)
tag = f"{TASK}_{MODALITY}_{MODEL}_{'combat' if WITH_COMBAT else 'nocombat'}"

print(f"Carregado: {RESULTS_PATH}")
print(
    f"Config: {TASK} | {MODALITY} | {MODEL} | combat={WITH_COMBAT} | "
    f"{EXPECTED_FOLDS} folds × {n_repeats} rep = {n_runs} avaliações"
)


## 2. Tabela agrupada

Ranqueada por **cobertura** → **amplitude** → nome.


In [ ]:
grp = feature_freq_table_grouped(df_cfg)
cols_show = [
    "feature_short",
    "coverage_pct",
    "pct_T1",
    "pct_T2",
    "pct_T3",
    "amplitude",
    "n_folds_any",
    "total_selections",
]
display(grp[cols_show])

principais = grp[grp["coverage_pct"] >= MIN_COVERAGE_PCT]
print(f"\nPrincipais (cobertura >= {MIN_COVERAGE_PCT}%): {len(principais)} / {len(grp)}")
display(principais[cols_show])


## 3. Gráfico principal

X = T1/T2/T3 · Y = % incidência · linha = biomarcador.


In [ ]:
lines_path = RESULTS_DIR / f"temporal_stability_{tag}.png"

fig = plot_temporal_stability_lines(
    df_cfg,
    out_path=lines_path if SAVE_OUTPUTS else None,
    sort_by="feature_short",
)
plt.show()
plt.close(fig)
if SAVE_OUTPUTS:
    print(f"Salvo: {lines_path}")


## 4. Comparação (colunas separadas) + export CSV


In [ ]:
compare_path = RESULTS_DIR / f"compare_stability_{tag}.png"
grouped_csv = RESULTS_DIR / f"feature_freq_grouped_{tag}.csv"
individual_csv = RESULTS_DIR / f"feature_freq_{tag}.csv"

fig = plot_compare_stability(df_cfg, out_path=compare_path if SAVE_OUTPUTS else None)
plt.show()
plt.close(fig)

if SAVE_OUTPUTS:
    grp.to_csv(grouped_csv, index=False)
    feature_freq_table(df_cfg).to_csv(individual_csv, index=False)
    print(f"Salvo: {compare_path}")
    print(f"Salvo: {grouped_csv}")
    print(f"Salvo: {individual_csv}")


## 5. (Opcional) Todas as configurações


In [ ]:
RUN_ALL_CONFIGS = False

if RUN_ALL_CONFIGS:
    for task_id in sorted(df_ablation["task"].astype(str).unique()):
        for model_key in sorted(df_ablation["model_key"].astype(str).unique()):
            for with_combat in sorted(df_ablation["with_combat"].unique()):
                try:
                    sub = filter_ablation_config(
                        df_ablation,
                        task=task_id,
                        modality=MODALITY,
                        model_key=model_key,
                        with_combat=bool(with_combat),
                        selection_mode=SELECTION_MODE,
                        expected_folds=EXPECTED_FOLDS,
                    )
                except ValueError as exc:
                    print(f"Pulando {task_id}/{model_key}/combat={with_combat}: {exc}")
                    continue
                t = f"{task_id}_{MODALITY}_{model_key}_{'combat' if with_combat else 'nocombat'}"
                plot_temporal_stability_lines(
                    sub, out_path=RESULTS_DIR / f"temporal_stability_{t}.png"
                )
                plot_compare_stability(
                    sub, out_path=RESULTS_DIR / f"compare_stability_{t}.png"
                )
                plt.close("all")
                print(f"OK: {t}")
